In [4]:
import pandas as pd
from pathlib import Path

# -------------------
# Config
# -------------------
# If your notebook/.py is in the project folder, leave BASE_DIR as "."
BASE_DIR = Path("C://Users//hasna//Downloads//Crypto vs Stock Market Diversification & Risk Insights Dashboard – SQL, Python, Power BI")                # or Path("crypto_project") if you have that folder
RAW_DIR = BASE_DIR / "data_raw"
OUT_DIR = BASE_DIR / "data_model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Map each CSV file (from Investing.com) to an asset symbol
files = {
    "BTC-USD_investing.csv": "BTC-USD",
    "ETH-USD_investing.csv": "ETH-USD",
    "SPY_investing.csv": "SPY",
    "GOLD_investing.csv": "GOLD",
}

all_prices = []

for filename, symbol in files.items():
    path = RAW_DIR / filename
    print(f"Processing: {path}")
    
    df = pd.read_csv(path)
    # Expect columns: Date, Price, Open, High, Low, Vol., Change %
    # Keep only Date + Price
    df = df[["Date", "Price"]].copy()
    df.rename(columns={"Price": "close_price"}, inplace=True)

    # --- Clean and convert close_price to numeric ---
    # Ensure string type first
    df["close_price"] = df["close_price"].astype(str)

    # Remove commas and currency symbols, and extra spaces
    df["close_price"] = (
        df["close_price"]
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("₹", "", regex=False)
        .str.strip()
    )

    # Convert to float; bad values become NaN
    df["close_price"] = pd.to_numeric(df["close_price"], errors="coerce")

    # Convert Date to datetime
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Attach symbol
    df["asset_symbol"] = symbol

    # Drop rows with missing date or price
    df = df.dropna(subset=["Date", "close_price"])

    # Remove zero / negative prices if any
    df = df[df["close_price"] > 0]

    # Optional: quick sanity check
    print(df.head(3))

    all_prices.append(df)

# -------------------
# Combine and compute daily returns
# -------------------
prices_long = pd.concat(all_prices, ignore_index=True)

# Sort by asset + date
prices_long = prices_long.sort_values(["asset_symbol", "Date"])

# Compute daily returns by group
prices_long["daily_return"] = (
    prices_long.groupby("asset_symbol")["close_price"]
    .pct_change()
)

# Rename Date -> price_date to match FRD naming
prices_long = prices_long.rename(columns={"Date": "price_date"})

# Split into two output tables
asset_prices = prices_long[["asset_symbol", "price_date", "close_price"]].copy()
asset_returns = prices_long[["asset_symbol", "price_date", "close_price", "daily_return"]].copy()

# Save to CSV
asset_prices.to_csv(OUT_DIR / "fact_asset_prices.csv", index=False)
asset_returns.to_csv(OUT_DIR / "fact_asset_returns.csv", index=False)

print("✅ Saved:")
print(" -", OUT_DIR / "fact_asset_prices.csv")
print(" -", OUT_DIR / "fact_asset_returns.csv")

Processing: C:\Users\hasna\Downloads\Crypto vs Stock Market Diversification & Risk Insights Dashboard – SQL, Python, Power BI\data_raw\BTC-USD_investing.csv
        Date  close_price asset_symbol
0 2025-12-01      86309.1      BTC-USD
1 2025-11-30      90374.2      BTC-USD
2 2025-11-29      90800.5      BTC-USD
Processing: C:\Users\hasna\Downloads\Crypto vs Stock Market Diversification & Risk Insights Dashboard – SQL, Python, Power BI\data_raw\ETH-USD_investing.csv
        Date  close_price asset_symbol
0 2025-12-01      2800.86      ETH-USD
1 2025-11-30      2991.73      ETH-USD
2 2025-11-29      2989.11      ETH-USD
Processing: C:\Users\hasna\Downloads\Crypto vs Stock Market Diversification & Risk Insights Dashboard – SQL, Python, Power BI\data_raw\SPY_investing.csv
        Date  close_price asset_symbol
0 2025-12-01       680.27          SPY
1 2025-11-28       683.39          SPY
2 2025-11-26       679.68          SPY
Processing: C:\Users\hasna\Downloads\Crypto vs Stock Market Diver